In [5]:
import os
import pandas as pd
import wfdb

# 1. Chapman folder ka rasta
CHAPMAN_PATH = (
    '../ecg-arrhythmia-1.0.0-chapman/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0/'
)

# 2. Bimariyon ki dictionary (CSV) dekho
snomed_df = pd.read_csv(CHAPMAN_PATH + 'ConditionNames_SNOMED-CT.csv')
print(snomed_df)

# 3. Pehle ECG (JS00001) ka header padh kar dekho usmein kya likha hai
rec = wfdb.rdrecord(CHAPMAN_PATH + 'WFDBRecords/01/010/JS00001')
print('\nSampling Rate:', rec.fs)
print('Shape:', rec.p_signal.shape)
print('Leads:', rec.sig_name)
print('Labels & Info:', rec.comments)


   Acronym Name                                     Full Name  Snomed_CT
0          1AVB               1 degree atrioventricular block  270492004
1          2AVB               2 degree atrioventricular block  195042002
2         2AVB1     2 degree atrioventricular block(Type one)   54016002
3         2AVB2     2 degree atrioventricular block(Type two)   28189009
4          3AVB               3 degree atrioventricular block   27885002
..          ...                                           ...        ...
58          SVT                  Supraventricular Tachycardia  426761007
59           AT                            Atrial Tachycardia  713422000
60        AVNRT  Atrioventricular  Node Reentrant Tachycardia  233896004
61         AVRT        Atrioventricular Reentrant Tachycardia  233897008
62        SAAWR       Sinus Atrium to Atrial Wandering Rhythm  195101003

[63 rows x 3 columns]

Sampling Rate: 500
Shape: (5000, 12)
Leads: ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3'

In [6]:
# Saari 63 conditions ek saath dekho taaki exact mapping bana sakein
for idx, row in snomed_df.iterrows():
  print(
      f"{idx:2d} | {str(row['Acronym Name']):8s} | {row['Snomed_CT']} |"
      f" {row['Full Name']}"
  )


 0 | 1AVB     | 270492004 | 1 degree atrioventricular block
 1 | 2AVB     | 195042002 | 2 degree atrioventricular block
 2 | 2AVB1    | 54016002 | 2 degree atrioventricular block(Type one)
 3 | 2AVB2    | 28189009 | 2 degree atrioventricular block(Type two)
 4 | 3AVB     | 27885002 | 3 degree atrioventricular block
 5 | ABI      | 251173003 | atrial bigeminy
 6 | ALS      | 39732003 | Axis left shift
 7 | APB      | 284470004 | atrial premature beats
 8 | AQW      | 164917005 | abnormal Q wave
 9 | ARS      | 47665007 | Axis right shift
10 | AVB      | 233917008 | atrioventricular block
11 | CCR      | 251199005 | countercolockwise rotation
12 | CR       | 251198002 | colockwise rotation
13 | ERV      | 428417006 | Early repolarization of the ventricles
14 | FQRS     | 164942001 | fQRS Wave
15 | IDC      | 698252002 | Interior differences conduction
16 | IVB      | 698252002 | Intraventricular block
17 | JEB      | 426995002 | junctional escape beat
18 | JPT      | 251164006 | junction

In [9]:
import glob
import os
import numpy as np
from scipy.signal import resample_poly
from sklearn.preprocessing import MultiLabelBinarizer
import wfdb

CHAPMAN_PATH = (
    '../ecg-arrhythmia-1.0.0-chapman/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0/'
)

SNOMED_TO_PTBXL = {
    '270492004': 'CD',
    '195042002': 'CD',
    '54016002': 'CD',
    '28189009': 'CD',
    '27885002': 'CD',
    '233917008': 'CD',
    '164909002': 'CD',
    '59118001': 'CD',
    '698252002': 'CD',
    '74390002': 'CD',
    '164873001': 'HYP',
    '89792004': 'HYP',
    '446358003': 'HYP',
    '164865005': 'MI',
    '428750005': 'STTC',
    '429622005': 'STTC',
    '164930006': 'STTC',
    '164931005': 'STTC',
    '164934002': 'STTC',
    '59931005': 'STTC',
    '111975006': 'STTC',
}

hea_files = sorted(
    glob.glob(
        os.path.join(CHAPMAN_PATH, 'WFDBRecords', '**', '*.hea'), recursive=True
    )
)
record_paths = [os.path.splitext(f) for f in hea_files]
print('Total .hea files found:', len(record_paths))

X_chap, y_chap_list = [], []
skipped_corrupt, skipped_no_diag = 0, 0

print('Chapman records load aur 100 Hz par resample ho rahe hain...')
for rec_path in record_paths:
  try:
    hdr = wfdb.rdheader(rec_path)

    dx_codes = []
    for c in hdr.comments:
      if c.startswith('Dx:'):
        dx_codes = [code.strip() for code in c.replace('Dx:', '').split(',')]
        break

    superclasses = set()
    for code in dx_codes:
      if code in SNOMED_TO_PTBXL:
        superclasses.add(SNOMED_TO_PTBXL[code])

    if len(superclasses) == 0 and '426783006' in dx_codes:
      superclasses.add('NORM')

    if len(superclasses) == 0:
      skipped_no_diag += 1
      continue

    rec = wfdb.rdrecord(rec_path)
    sig = rec.p_signal.astype(np.float32)

    if sig.shape != (5000, 12) or np.isnan(sig).any() or np.isinf(sig).any():
      skipped_corrupt += 1
      continue

    sig_100hz = resample_poly(sig, up=1, down=5, axis=0).astype(np.float32)
    X_chap.append(sig_100hz)
    y_chap_list.append(list(superclasses))

  except Exception:
    skipped_corrupt += 1
    continue

X_chap = np.array(X_chap, dtype=np.float32)

mlb = MultiLabelBinarizer(classes=['CD', 'HYP', 'MI', 'NORM', 'STTC'])
y_chap = mlb.fit_transform(y_chap_list)

os.makedirs('processed_chapman', exist_ok=True)
np.save('processed_chapman/X_chapman.npy', X_chap)
np.save('processed_chapman/y_chapman.npy', y_chap)

print('\nChapman Processing Complete!')
print('Clean Chapman Shape :', X_chap.shape)
print(
    f'Skipped (Pure Rhythm/No-Diag): {skipped_no_diag} | Skipped'
    f' (Corrupt/Bad-Date): {skipped_corrupt}'
)
print('\nClass Distribution in Chapman:')
for i, c in enumerate(mlb.classes_):
  print(f'  {c:4s}: {y_chap[:, i].sum():5d} ({y_chap[:, i].mean()*100:5.1f}%)')


Total .hea files found: 45152
Chapman records load aur 100 Hz par resample ho rahe hain...

Chapman Processing Complete!
Clean Chapman Shape : (0,)
Skipped (Pure Rhythm/No-Diag): 0 | Skipped (Corrupt/Bad-Date): 45152

Class Distribution in Chapman:
  CD  :     0 (  nan%)
  HYP :     0 (  nan%)
  MI  :     0 (  nan%)
  NORM:     0 (  nan%)
  STTC:     0 (  nan%)


C:\Users\DELL-DAEMON\AppData\Local\Temp\ipykernel_25468\871374835.py:102: RuntimeWarning: Mean of empty slice
  print(f'  {c:4s}: {y_chap[:, i].sum():5d} ({y_chap[:, i].mean()*100:5.1f}%)')
c:\Users\DELL-DAEMON\Documents\Github\XAI_Cardio_Research\coding\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [11]:
import os
import scipy.io

sample_hea = next(iter(hea_files))
sample_mat = sample_hea.replace('.hea', '.mat')

print('--- RAW .hea FILE ---')
with open(sample_hea, 'r') as f:
  print(f.read())

mat = scipy.io.loadmat(sample_mat)
print('--- .mat KEYS & SHAPE ---')
for k, v in mat.items():
  if not k.startswith('__'):
    print(k, v.shape, v.dtype)


--- RAW .hea FILE ---
JS00001 12 500 5000
JS00001.mat 16+24 1000/mV 16 0 -254 21756 0 I
JS00001.mat 16+24 1000/mV 16 0 264 -599 0 II
JS00001.mat 16+24 1000/mV 16 0 517 -22376 0 III
JS00001.mat 16+24 1000/mV 16 0 -5 28232 0 aVR
JS00001.mat 16+24 1000/mV 16 0 -386 16619 0 aVL
JS00001.mat 16+24 1000/mV 16 0 390 15121 0 aVF
JS00001.mat 16+24 1000/mV 16 0 -98 1568 0 V1
JS00001.mat 16+24 1000/mV 16 0 -312 -32761 0 V2
JS00001.mat 16+24 1000/mV 16 0 -98 32715 0 V3
JS00001.mat 16+24 1000/mV 16 0 810 15193 0 V4
JS00001.mat 16+24 1000/mV 16 0 810 14081 0 V5
JS00001.mat 16+24 1000/mV 16 0 527 32579 0 V6
#Age: 85
#Sex: Male
#Dx: 164889003,59118001,164934002
#Rx: Unknown
#Hx: Unknown
#Sx: Unknown

--- .mat KEYS & SHAPE ---
val (12, 5000) int16


In [12]:
import glob
import os
import numpy as np
import scipy.io
from scipy.signal import resample_poly
from sklearn.preprocessing import MultiLabelBinarizer

CHAPMAN_PATH = (
    '../ecg-arrhythmia-1.0.0-chapman/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0/'
)

# 1. SNOMED-CT -> PTB-XL 5 Superclasses Dictionary
SNOMED_TO_PTBXL = {
    '270492004': 'CD',
    '195042002': 'CD',
    '54016002': 'CD',
    '28189009': 'CD',
    '27885002': 'CD',
    '233917008': 'CD',
    '164909002': 'CD',
    '59118001': 'CD',
    '698252002': 'CD',
    '74390002': 'CD',
    '164873001': 'HYP',
    '89792004': 'HYP',
    '446358003': 'HYP',
    '164865005': 'MI',
    '428750005': 'STTC',
    '429622005': 'STTC',
    '164930006': 'STTC',
    '164931005': 'STTC',
    '164934002': 'STTC',
    '59931005': 'STTC',
    '111975006': 'STTC',
}

# 2. Saari .hea files ki list lo
hea_files = sorted(
    glob.glob(
        os.path.join(CHAPMAN_PATH, 'WFDBRecords', '**', '*.hea'), recursive=True
    )
)
print('Total .hea files found:', len(hea_files))

X_chap, y_chap_list = [], []
skipped_corrupt, skipped_no_diag = 0, 0

print('Fast loading & resampling to 100 Hz (bina wfdb ke, ~30 sec lagenge)...')
for hea_path in hea_files:
  # Direct text file se #Dx line padho
  dx_codes = []
  with open(hea_path, 'r', errors='ignore') as f:
    for line in f:
      if line.startswith('#Dx:'):
        dx_codes = [
            c.strip() for c in line.replace('#Dx:', '').split(',') if c.strip()
        ]
        break

  # Map to 5 superclasses
  superclasses = set()
  for code in dx_codes:
    if code in SNOMED_TO_PTBXL:
      superclasses.add(SNOMED_TO_PTBXL[code])

  if len(superclasses) == 0 and '426783006' in dx_codes:
    superclasses.add('NORM')

  if len(superclasses) == 0:
    skipped_no_diag += 1
    continue

  # Direct .mat file load karo aur mV mein badlo (/ 1000.0)
  mat_path = hea_path.replace('.hea', '.mat')
  try:
    sig = scipy.io.loadmat(mat_path)['val'].astype(np.float32) / 1000.0
  except Exception:
    skipped_corrupt += 1
    continue

  # Shape (12, 5000) check + NaN/Inf check
  if sig.shape != (12, 5000) or np.isnan(sig).any() or np.isinf(sig).any():
    skipped_corrupt += 1
    continue

  # 500 Hz -> 100 Hz downsample (axis=1 kyunki time axis 1 hai) -> (12, 1000)
  sig_100hz = resample_poly(sig, up=1, down=5, axis=1).astype(np.float32)
  X_chap.append(sig_100hz)
  y_chap_list.append(list(superclasses))

X_chap = np.array(
    X_chap, dtype=np.float32
)  # Shape: (N, 12, 1000) - PyTorch ke liye direct ready!

# 3. Exact PTB-XL order ['CD', 'HYP', 'MI', 'NORM', 'STTC'] mein encode karo
mlb = MultiLabelBinarizer(classes=['CD', 'HYP', 'MI', 'NORM', 'STTC'])
y_chap = mlb.fit_transform(y_chap_list)

# 4. Save kar lo!
os.makedirs('processed_chapman', exist_ok=True)
np.save('processed_chapman/X_chapman.npy', X_chap)
np.save('processed_chapman/y_chapman.npy', y_chap)

print('\nChapman Processing Complete!')
print('Clean Chapman Shape (PyTorch ready):', X_chap.shape)
print(
    f'Skipped (Pure Rhythm/No-Diag): {skipped_no_diag} | Skipped (Corrupt/NaN):'
    f' {skipped_corrupt}'
)
print('\nClass Distribution in Chapman:')
for i, c in enumerate(mlb.classes_):
  print(f'  {c:4s}: {y_chap[:, i].sum():5d} ({y_chap[:, i].mean()*100:5.1f}%)')


Total .hea files found: 45152
Fast loading & resampling to 100 Hz (bina wfdb ke, ~30 sec lagenge)...

Chapman Processing Complete!
Clean Chapman Shape (PyTorch ready): (19739, 12, 1000)
Skipped (Pure Rhythm/No-Diag): 25413 | Skipped (Corrupt/NaN): 0

Class Distribution in Chapman:
  CD  :  3127 ( 15.8%)
  HYP :   769 (  3.9%)
  MI  :   123 (  0.6%)
  NORM:  6973 ( 35.3%)
  STTC: 10358 ( 52.5%)


In [14]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classes = ['CD', 'HYP', 'MI', 'NORM', 'STTC']

# 1. PTB-XL Train ka mean/std nikaalo (exact same scaling ke liye)
X_ptb_train = np.load('processed_data/X_train.npy')  # (17111, 1000, 12)
mean_12 = X_ptb_train.mean(axis=(0, 1)).reshape(1, 12, 1)
std_12 = X_ptb_train.std(axis=(0, 1)).reshape(1, 12, 1) + 1e-8

# 2. Chapman load aur scale karo
X_chap = np.load('processed_chapman/X_chapman.npy')  # (19739, 12, 1000)
y_chap = np.load('processed_chapman/y_chapman.npy')  # (19739, 5)
X_chap_norm = (X_chap - mean_12) / std_12

chap_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_chap_norm, dtype=torch.float32),
        torch.tensor(y_chap, dtype=torch.float32),
    ),
    batch_size=64,
    shuffle=False,
)


# 3. Apna InceptionNet architecture define karo aur saved weights load karo
class InceptionModule(nn.Module):

  def __init__(self, in_ch, nf=32, bottleneck=32, ks=(9, 19, 39)):
    super().__init__()
    self.bottleneck = nn.Conv1d(in_ch, bottleneck, 1, bias=False)
    self.convs = nn.ModuleList(
        [nn.Conv1d(bottleneck, nf, k, padding='same', bias=False) for k in ks]
    )
    self.pool_branch = nn.Sequential(
        nn.MaxPool1d(3, stride=1, padding=1),
        nn.Conv1d(in_ch, nf, 1, bias=False),
    )
    self.bn = nn.BatchNorm1d(nf * (len(ks) + 1))

  def forward(self, x):
    z = self.bottleneck(x)
    out = torch.cat([c(z) for c in self.convs] + [self.pool_branch(x)], dim=1)
    return torch.relu(self.bn(out))


class InceptionNet(nn.Module):

  def __init__(self, in_ch=12, n_classes=5, nf=32, depth=6):
    super().__init__()
    out_ch = nf * 4
    self.blocks = nn.ModuleList([
        InceptionModule(in_ch if d == 0 else out_ch, nf) for d in range(depth)
    ])
    self.shortcuts = nn.ModuleList([
        nn.Sequential(
            nn.Conv1d(in_ch if i == 0 else out_ch, out_ch, 1, bias=False),
            nn.BatchNorm1d(out_ch),
        )
        for i in range(depth // 3)
    ])
    self.fc = nn.Linear(out_ch, n_classes)

  def forward(self, x):
    res, r = x, 0
    for d, blk in enumerate(self.blocks):
      x = blk(x)
      if d % 3 == 2:
        x = torch.relu(x + self.shortcuts[r](res))
        res, r = x, r + 1
    return self.fc(x.mean(dim=-1))


# Saved PTB-XL weights load karo!
model = InceptionNet(n_classes=5).to(device)
model.load_state_dict(torch.load('inception_seed42.pt', map_location=device))
model.eval()

# 4. Chapman par predictions nikaalo
chap_probs = []
with torch.no_grad():
  for xb, _ in chap_loader:
    chap_probs.append(torch.sigmoid(model(xb.to(device))).cpu().numpy())
pt_chap = np.vstack(chap_probs)

# 5. PTB-XL Fold 9 se tuned thresholds (jo tumhare Inception1D mein aaye the)
ptb_thresholds = [0.82, 0.75, 0.71, 0.68, 0.53]

rows = []
for i, c in enumerate(classes):
  yt = y_chap[:, i].astype(bool)
  yp = pt_chap[:, i] >= ptb_thresholds[i]
  tp, fn = (yp & yt).sum(), (~yp & yt).sum()
  tn, fp = (~yp & ~yt).sum(), (yp & ~yt).sum()
  rows.append(
      dict(
          class_=c,
          n_pos=int(yt.sum()),
          AUROC=roc_auc_score(yt, pt_chap[:, i]),
          AUPRC=average_precision_score(yt, pt_chap[:, i]),
          F1=f1_score(yt, yp, zero_division=0),
          sens=tp / (tp + fn),
          spec=tn / (tn + fp),
      )
  )

res_chap = pd.DataFrame(rows).set_index('class_')
print('=== EXTERNAL VALIDATION ON CHAPMAN (ZERO-SHOT FROM PTB-XL) ===')
print(res_chap.round(3).to_string())
print(
    f'\nCHAPMAN Macro-AUROC: {res_chap.AUROC.mean():.4f} | Macro-AUPRC:'
    f' {res_chap.AUPRC.mean():.4f} | Macro-F1: {res_chap.F1.mean():.4f}'
)


=== EXTERNAL VALIDATION ON CHAPMAN (ZERO-SHOT FROM PTB-XL) ===
        n_pos  AUROC  AUPRC     F1   sens   spec
class_                                          
CD       3127  0.830  0.456  0.476  0.528  0.869
HYP       769  0.921  0.380  0.256  0.867  0.801
MI        123  0.887  0.087  0.044  0.870  0.761
NORM     6973  0.929  0.875  0.809  0.790  0.910
STTC    10358  0.853  0.839  0.772  0.719  0.840

CHAPMAN Macro-AUROC: 0.8842 | Macro-AUPRC: 0.5273 | Macro-F1: 0.4711


In [15]:
import torch

ckpt_files = [
    'best_simple_cnn.pth',
    'inception_seed42.pt',
    'se_inception_seed42.pt',
    'se_temporal_inception_seed42.pt',
    'inc_transformer_seed42.pt',
]

for f in ckpt_files:
  try:
    sd = torch.load(f, map_location='cpu')
    keys = list(sd.keys())
    print(f'=== {f} (total tensors: {len(keys)}) ===')
    print('  First 6 keys:', keys[:6])
    print('  Last  4 keys:', keys[-4:])
  except Exception as e:
    print(f'=== {f}: {e} ===')


=== best_simple_cnn.pth (total tensors: 23) ===
  First 6 keys: ['conv1.0.weight', 'conv1.0.bias', 'conv1.1.weight', 'conv1.1.bias', 'conv1.1.running_mean', 'conv1.1.running_var']
  Last  4 keys: ['conv3.1.running_var', 'conv3.1.num_batches_tracked', 'fc.weight', 'fc.bias']
=== inception_seed42.pt (total tensors: 74) ===
  First 6 keys: ['blocks.0.bottleneck.weight', 'blocks.0.convs.0.weight', 'blocks.0.convs.1.weight', 'blocks.0.convs.2.weight', 'blocks.0.pool_branch.1.weight', 'blocks.0.bn.weight']
  Last  4 keys: ['shortcuts.1.1.running_var', 'shortcuts.1.1.num_batches_tracked', 'fc.weight', 'fc.bias']
=== se_inception_seed42.pt (total tensors: 86) ===
  First 6 keys: ['blocks.0.bottleneck.weight', 'blocks.0.convs.0.weight', 'blocks.0.convs.1.weight', 'blocks.0.convs.2.weight', 'blocks.0.pool_branch.1.weight', 'blocks.0.bn.weight']
  Last  4 keys: ['shortcuts.1.1.running_var', 'shortcuts.1.1.num_batches_tracked', 'fc.weight', 'fc.bias']
=== se_temporal_inception_seed42.pt (total ten

In [16]:
import torch

for f in [
    'se_inception_seed42.pt',
    'se_temporal_inception_seed42.pt',
    'inc_transformer_seed42.pt',
]:
  sd = torch.load(f, map_location='cpu')
  print(f'=== {f} ===')
  for k, v in sd.items():
    if 'weight' in k and 'bn' not in k and 'norm' not in k:
      print(f'  {k}: {tuple(v.shape)}')


=== se_inception_seed42.pt ===
  blocks.0.bottleneck.weight: (32, 12, 1)
  blocks.0.convs.0.weight: (32, 32, 9)
  blocks.0.convs.1.weight: (32, 32, 19)
  blocks.0.convs.2.weight: (32, 32, 39)
  blocks.0.pool_branch.1.weight: (32, 12, 1)
  blocks.0.se.excite.0.weight: (16, 128)
  blocks.0.se.excite.2.weight: (128, 16)
  blocks.1.bottleneck.weight: (32, 128, 1)
  blocks.1.convs.0.weight: (32, 32, 9)
  blocks.1.convs.1.weight: (32, 32, 19)
  blocks.1.convs.2.weight: (32, 32, 39)
  blocks.1.pool_branch.1.weight: (32, 128, 1)
  blocks.1.se.excite.0.weight: (16, 128)
  blocks.1.se.excite.2.weight: (128, 16)
  blocks.2.bottleneck.weight: (32, 128, 1)
  blocks.2.convs.0.weight: (32, 32, 9)
  blocks.2.convs.1.weight: (32, 32, 19)
  blocks.2.convs.2.weight: (32, 32, 39)
  blocks.2.pool_branch.1.weight: (32, 128, 1)
  blocks.2.se.excite.0.weight: (16, 128)
  blocks.2.se.excite.2.weight: (128, 16)
  blocks.3.bottleneck.weight: (32, 128, 1)
  blocks.3.convs.0.weight: (32, 32, 9)
  blocks.3.convs.1.

In [17]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cpu')  # Zero VRAM crash guarantee (~45s total on CPU)
classes = ['CD', 'HYP', 'MI', 'NORM', 'STTC']

# ==========================================================
# 1. LOAD & STANDARDIZE PTB-XL (Val/Test) & CHAPMAN
# ==========================================================
X_train = np.load('processed_data/X_train.npy')
mean_12 = X_train.mean(axis=(0, 1)).reshape(1, 12, 1)
std_12 = X_train.std(axis=(0, 1)).reshape(1, 12, 1) + 1e-8

X_val = (
    np.transpose(np.load('processed_data/X_val.npy'), (0, 2, 1)) - mean_12
) / std_12
y_val = np.load('processed_data/y_val.npy')

X_test = (
    np.transpose(np.load('processed_data/X_test.npy'), (0, 2, 1)) - mean_12
) / std_12
y_test = np.load('processed_data/y_test.npy')

X_chap = (np.load('processed_chapman/X_chapman.npy') - mean_12) / std_12
y_chap = np.load('processed_chapman/y_chapman.npy')

val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val, dtype=torch.float32)), batch_size=128
)
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test, dtype=torch.float32)), batch_size=128
)
chap_loader = DataLoader(
    TensorDataset(torch.tensor(X_chap, dtype=torch.float32)), batch_size=128
)


# ==========================================================
# 2. EXACT ARCHITECTURES FOR ALL 5 SAVED CHECKPOINTS
# ==========================================================
# Model 1: SimpleECGCNN
class SimpleECGCNN(nn.Module):

  def __init__(self, num_classes=5):
    super().__init__()
    self.conv1 = nn.Sequential(
        nn.Conv1d(12, 32, 7, 2, 3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2)
    )
    self.conv2 = nn.Sequential(
        nn.Conv1d(32, 64, 5, 2, 2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2)
    )
    self.conv3 = nn.Sequential(
        nn.Conv1d(64, 128, 3, 1, 1),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.AdaptiveAvgPool1d(1),
    )
    self.fc = nn.Linear(128, num_classes)

  def forward(self, x):
    return self.fc(self.conv3(self.conv2(self.conv1(x))).squeeze(-1))


# Shared Inception Building Blocks
class SEBlock1D(nn.Module):

  def __init__(self, ch, r=8):
    super().__init__()
    self.excite = nn.Sequential(
        nn.Linear(ch, ch // r),
        nn.ReLU(),
        nn.Linear(ch // r, ch),
        nn.Sigmoid(),
    )

  def forward(self, x):
    w = self.excite(x.mean(dim=-1)).unsqueeze(-1)
    return x * w


class InceptionModule(nn.Module):

  def __init__(self, in_ch, nf=32, bottleneck=32, ks=(9, 19, 39), use_se=False):
    super().__init__()
    self.bottleneck = nn.Conv1d(in_ch, bottleneck, 1, bias=False)
    self.convs = nn.ModuleList(
        [nn.Conv1d(bottleneck, nf, k, padding='same', bias=False) for k in ks]
    )
    self.pool_branch = nn.Sequential(
        nn.MaxPool1d(3, stride=1, padding=1),
        nn.Conv1d(in_ch, nf, 1, bias=False),
    )
    self.bn = nn.BatchNorm1d(nf * (len(ks) + 1))
    self.se = SEBlock1D(nf * (len(ks) + 1)) if use_se else nn.Identity()

  def forward(self, x):
    z = self.bottleneck(x)
    out = torch.cat([c(z) for c in self.convs] + [self.pool_branch(x)], dim=1)
    return self.se(torch.relu(self.bn(out)))


class TemporalAttention1D(nn.Module):

  def __init__(self, ch):
    super().__init__()
    self.attn = nn.Sequential(
        nn.Conv1d(ch, ch // 2, 1), nn.Tanh(), nn.Conv1d(ch // 2, 1, 1)
    )

  def forward(self, x):
    w = torch.softmax(self.attn(x), dim=-1)
    return (x * w).sum(dim=-1)


# Models 2, 3, 4: InceptionNet / SE-InceptionNet / SE-Temporal-InceptionNet
class InceptionNet(nn.Module):

  def __init__(
      self,
      in_ch=12,
      n_classes=5,
      nf=32,
      depth=6,
      use_se=False,
      use_temp_attn=False,
  ):
    super().__init__()
    out_ch = nf * 4
    self.blocks = nn.ModuleList([
        InceptionModule(in_ch if d == 0 else out_ch, nf, use_se=use_se)
        for d in range(depth)
    ])
    self.shortcuts = nn.ModuleList([
        nn.Sequential(
            nn.Conv1d(in_ch if i == 0 else out_ch, out_ch, 1, bias=False),
            nn.BatchNorm1d(out_ch),
        )
        for i in range(depth // 3)
    ])
    self.temporal_attn = (
        TemporalAttention1D(out_ch) if use_temp_attn else None
    )
    self.fc = nn.Linear(out_ch, n_classes)

  def forward(self, x):
    res, r = x, 0
    for d, blk in enumerate(self.blocks):
      x = blk(x)
      if d % 3 == 2:
        x = torch.relu(x + self.shortcuts[r](res))
        res, r = x, r + 1
    feat = (
        self.temporal_attn(x)
        if self.temporal_attn is not None
        else x.mean(dim=-1)
    )
    return self.fc(feat)


# Model 5: Inception-Transformer Hybrid
class InceptionTransformer(nn.Module):

  def __init__(self, in_ch=12, n_classes=5, nf=32):
    super().__init__()
    out_ch = nf * 4
    self.inc1 = InceptionModule(in_ch, nf)
    self.inc2 = InceptionModule(out_ch, nf)
    self.inc3 = InceptionModule(out_ch, nf)
    self.shortcut = nn.Sequential(
        nn.Conv1d(in_ch, out_ch, 1, bias=False), nn.BatchNorm1d(out_ch)
    )
    enc_layer = nn.TransformerEncoderLayer(
        d_model=out_ch, nhead=4, dim_feedforward=256, batch_first=True
    )
    self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)
    self.fc = nn.Linear(out_ch, n_classes)

  def forward(self, x):
    res = self.shortcut(x)
    x = torch.relu(self.inc3(self.inc2(self.inc1(x))) + res)
    x = nn.functional.adaptive_avg_pool1d(x, 125).transpose(
        1, 2
    )  # (B, 125, 128)
    x = self.transformer(x).mean(dim=1)
    return self.fc(x)


# ==========================================================
# 3. EVALUATE ALL 5 MODELS SIDE-BY-SIDE
# ==========================================================
models_to_eval = [
    ('1. Baseline 1D-CNN', SimpleECGCNN(), 'best_simple_cnn.pth'),
    ('2. Inception1D', InceptionNet(), 'inception_seed42.pt'),
    (
        '3. SE-Inception1D (Lead Attn)',
        InceptionNet(use_se=True),
        'se_inception_seed42.pt',
    ),
    (
        '4. SE-Temporal-Inception1D',
        InceptionNet(use_se=True, use_temp_attn=True),
        'se_temporal_inception_seed42.pt',
    ),
    (
        '5. Inception-Transformer',
        InceptionTransformer(),
        'inc_transformer_seed42.pt',
    ),
]


@torch.no_grad()
def get_probs(m, loader):
  m.eval()
  return np.vstack([torch.sigmoid(m(xb)).numpy() for (xb,) in loader])


summary = []
per_class_chap = {}

for name, m, ckpt in models_to_eval:
  try:
    m.load_state_dict(torch.load(ckpt, map_location=device))
    params = sum(p.numel() for p in m.parameters())

    pv = get_probs(m, val_loader)
    pt = get_probs(m, test_loader)
    pc = get_probs(m, chap_loader)

    # Tune thresholds on PTB-XL Fold 9
    ths = np.array([
        np.arange(0.05, 0.96, 0.01)[
            int(
                np.argmax([
                    f1_score(y_val[:, i], pv[:, i] >= t, zero_division=0)
                    for t in np.arange(0.05, 0.96, 0.01)
                ])
            )
        ]
        for i in range(5)
    ])

    ptb_auc = roc_auc_score(y_test, pt, average='macro')
    ptb_f1 = f1_score(y_test, pt >= ths, average='macro', zero_division=0)

    chap_auc = roc_auc_score(y_chap, pc, average='macro')
    chap_f1 = f1_score(y_chap, pc >= ths, average='macro', zero_division=0)
    chap_auprc = average_precision_score(y_chap, pc, average='macro')

    summary.append({
        'Model': name,
        'Params': params,
        'PTB_AUROC': ptb_auc,
        'PTB_F1': ptb_f1,
        'CHAP_AUROC': chap_auc,
        'CHAP_AUPRC': chap_auprc,
        'CHAP_F1': chap_f1,
        'AUROC_Drop': ptb_auc - chap_auc,
    })
    per_class_chap[name] = [
        roc_auc_score(y_chap[:, i], pc[:, i]) for i in range(5)
    ]
    print(
        f'Done: {name} -> PTB AUROC: {ptb_auc:.4f} | Chapman AUROC:'
        f' {chap_auc:.4f}'
    )
  except Exception as e:
    print(f'Error in {name}: {e}')

df_summary = pd.DataFrame(summary).set_index('Model')
print('\n================ MASTER CROSS-DATASET COMPARISON ================')
print(df_summary.round(4).to_string())

print('\n================ CHAPMAN PER-CLASS AUROC ================')
print(
    pd.DataFrame.from_dict(
        per_class_chap, orient='index', columns=classes
    ).round(4).to_string()
)


Done: 1. Baseline 1D-CNN -> PTB AUROC: 0.9189 | Chapman AUROC: 0.8794
Done: 2. Inception1D -> PTB AUROC: 0.9277 | Chapman AUROC: 0.8842
Error in 3. SE-Inception1D (Lead Attn): Error(s) in loading state_dict for InceptionNet:
	Missing key(s) in state_dict: "blocks.0.se.excite.0.bias", "blocks.0.se.excite.2.bias", "blocks.1.se.excite.0.bias", "blocks.1.se.excite.2.bias", "blocks.2.se.excite.0.bias", "blocks.2.se.excite.2.bias", "blocks.3.se.excite.0.bias", "blocks.3.se.excite.2.bias", "blocks.4.se.excite.0.bias", "blocks.4.se.excite.2.bias", "blocks.5.se.excite.0.bias", "blocks.5.se.excite.2.bias". 
Error in 4. SE-Temporal-Inception1D: Error(s) in loading state_dict for InceptionNet:
	Missing key(s) in state_dict: "blocks.0.se.excite.0.bias", "blocks.0.se.excite.2.bias", "blocks.1.se.excite.0.bias", "blocks.1.se.excite.2.bias", "blocks.2.se.excite.0.bias", "blocks.2.se.excite.2.bias", "blocks.3.se.excite.0.bias", "blocks.3.se.excite.2.bias", "blocks.4.se.excite.0.bias", "blocks.4.se.exci

In [18]:
import math


# Fix 1: SEBlock1D with bias=False
class SEBlock1D(nn.Module):

  def __init__(self, ch, r=8):
    super().__init__()
    self.excite = nn.Sequential(
        nn.Linear(ch, ch // r, bias=False),
        nn.ReLU(),
        nn.Linear(ch // r, ch, bias=False),
        nn.Sigmoid(),
    )

  def forward(self, x):
    w = self.excite(x.mean(dim=-1)).unsqueeze(-1)
    return x * w


# Re-define InceptionModule & InceptionNet using the fixed SEBlock1D
class InceptionModule(nn.Module):

  def __init__(self, in_ch, nf=32, bottleneck=32, ks=(9, 19, 39), use_se=False):
    super().__init__()
    self.bottleneck = nn.Conv1d(in_ch, bottleneck, 1, bias=False)
    self.convs = nn.ModuleList(
        [nn.Conv1d(bottleneck, nf, k, padding='same', bias=False) for k in ks]
    )
    self.pool_branch = nn.Sequential(
        nn.MaxPool1d(3, stride=1, padding=1),
        nn.Conv1d(in_ch, nf, 1, bias=False),
    )
    self.bn = nn.BatchNorm1d(nf * (len(ks) + 1))
    self.se = SEBlock1D(nf * (len(ks) + 1)) if use_se else nn.Identity()

  def forward(self, x):
    z = self.bottleneck(x)
    out = torch.cat([c(z) for c in self.convs] + [self.pool_branch(x)], dim=1)
    return self.se(torch.relu(self.bn(out)))


class InceptionNet(nn.Module):

  def __init__(
      self,
      in_ch=12,
      n_classes=5,
      nf=32,
      depth=6,
      use_se=False,
      use_temp_attn=False,
  ):
    super().__init__()
    out_ch = nf * 4
    self.blocks = nn.ModuleList([
        InceptionModule(in_ch if d == 0 else out_ch, nf, use_se=use_se)
        for d in range(depth)
    ])
    self.shortcuts = nn.ModuleList([
        nn.Sequential(
            nn.Conv1d(in_ch if i == 0 else out_ch, out_ch, 1, bias=False),
            nn.BatchNorm1d(out_ch),
        )
        for i in range(depth // 3)
    ])
    self.temporal_attn = (
        TemporalAttention1D(out_ch) if use_temp_attn else None
    )
    self.fc = nn.Linear(out_ch, n_classes)

  def forward(self, x):
    res, r = x, 0
    for d, blk in enumerate(self.blocks):
      x = blk(x)
      if d % 3 == 2:
        x = torch.relu(x + self.shortcuts[r](res))
        res, r = x, r + 1
    feat = (
        self.temporal_attn(x)
        if self.temporal_attn is not None
        else x.mean(dim=-1)
    )
    return self.fc(feat)


# Fix 2: PositionalEncoding + InceptionTransformer
class PositionalEncoding(nn.Module):

  def __init__(self, d_model, max_len=2000):
    super().__init__()
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float32)
        * (-math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    self.register_buffer('pe', pe.unsqueeze(0))

  def forward(self, x):
    return x + self.pe[:, : x.size(1)]


class InceptionTransformer(nn.Module):

  def __init__(self, in_ch=12, n_classes=5, nf=32):
    super().__init__()
    out_ch = nf * 4
    self.inc1 = InceptionModule(in_ch, nf)
    self.inc2 = InceptionModule(out_ch, nf)
    self.inc3 = InceptionModule(out_ch, nf)
    self.shortcut = nn.Sequential(
        nn.Conv1d(in_ch, out_ch, 1, bias=False), nn.BatchNorm1d(out_ch)
    )
    self.pos_enc = PositionalEncoding(out_ch)
    enc_layer = nn.TransformerEncoderLayer(
        d_model=out_ch, nhead=4, dim_feedforward=256, batch_first=True
    )
    self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)
    self.fc = nn.Linear(out_ch, n_classes)

  def forward(self, x):
    res = self.shortcut(x)
    x = torch.relu(self.inc3(self.inc2(self.inc1(x))) + res)
    x = nn.functional.adaptive_avg_pool1d(x, 125).transpose(1, 2)
    x = self.transformer(self.pos_enc(x)).mean(dim=1)
    return self.fc(x)


# Evaluate all 5 models cleanly!
models_to_eval = [
    ('1. Baseline 1D-CNN', SimpleECGCNN(), 'best_simple_cnn.pth'),
    ('2. Inception1D', InceptionNet(), 'inception_seed42.pt'),
    (
        '3. SE-Inception1D (Lead Attn)',
        InceptionNet(use_se=True),
        'se_inception_seed42.pt',
    ),
    (
        '4. SE-Temporal-Inception1D',
        InceptionNet(use_se=True, use_temp_attn=True),
        'se_temporal_inception_seed42.pt',
    ),
    (
        '5. Inception-Transformer',
        InceptionTransformer(),
        'inc_transformer_seed42.pt',
    ),
]

summary, per_class_chap = [], {}
for name, m, ckpt in models_to_eval:
  m.load_state_dict(torch.load(ckpt, map_location=device))
  params = sum(p.numel() for p in m.parameters())

  pv, pt, pc = (
      get_probs(m, val_loader),
      get_probs(m, test_loader),
      get_probs(m, chap_loader),
  )
  ths = np.array([
      np.arange(0.05, 0.96, 0.01)[
          int(
              np.argmax([
                  f1_score(y_val[:, i], pv[:, i] >= t, zero_division=0)
                  for t in np.arange(0.05, 0.96, 0.01)
              ])
          )
      ]
      for i in range(5)
  ])

  ptb_auc = roc_auc_score(y_test, pt, average='macro')
  ptb_f1 = f1_score(y_test, pt >= ths, average='macro', zero_division=0)
  chap_auc = roc_auc_score(y_chap, pc, average='macro')
  chap_f1 = f1_score(y_chap, pc >= ths, average='macro', zero_division=0)
  chap_auprc = average_precision_score(y_chap, pc, average='macro')

  summary.append({
      'Model': name,
      'Params': params,
      'PTB_AUROC': ptb_auc,
      'PTB_F1': ptb_f1,
      'CHAP_AUROC': chap_auc,
      'CHAP_AUPRC': chap_auprc,
      'CHAP_F1': chap_f1,
      'AUROC_Drop': ptb_auc - chap_auc,
  })
  per_class_chap[name] = [
      roc_auc_score(y_chap[:, i], pc[:, i]) for i in range(5)
  ]
  print(
      f'Done: {name} -> PTB AUROC: {ptb_auc:.4f} | Chapman AUROC:'
      f' {chap_auc:.4f}'
  )

print('\n================ MASTER CROSS-DATASET COMPARISON ================')
print(pd.DataFrame(summary).set_index('Model').round(4).to_string())

print('\n================ CHAPMAN PER-CLASS AUROC ================')
print(
    pd.DataFrame.from_dict(
        per_class_chap, orient='index', columns=classes
    ).round(4).to_string()
)


Done: 1. Baseline 1D-CNN -> PTB AUROC: 0.9189 | Chapman AUROC: 0.8794
Done: 2. Inception1D -> PTB AUROC: 0.9277 | Chapman AUROC: 0.8842
Done: 3. SE-Inception1D (Lead Attn) -> PTB AUROC: 0.9298 | Chapman AUROC: 0.8829
Done: 4. SE-Temporal-Inception1D -> PTB AUROC: 0.9261 | Chapman AUROC: 0.8867


RuntimeError: Error(s) in loading state_dict for InceptionTransformer:
	size mismatch for pos_enc.pe: copying a param with shape torch.Size([1, 200, 128]) from checkpoint, the shape in current model is torch.Size([1, 2000, 128]).

In [19]:
class PositionalEncoding(nn.Module):

  def __init__(self, d_model, max_len=200):  # <-- 200 matches saved checkpoint!
    super().__init__()
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float32)
        * (-math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    self.register_buffer('pe', pe.unsqueeze(0))

  def forward(self, x):
    return x + self.pe[:, : x.size(1)]


class InceptionTransformer(nn.Module):

  def __init__(self, in_ch=12, n_classes=5, nf=32):
    super().__init__()
    out_ch = nf * 4
    self.inc1 = InceptionModule(in_ch, nf)
    self.inc2 = InceptionModule(out_ch, nf)
    self.inc3 = InceptionModule(out_ch, nf)
    self.shortcut = nn.Sequential(
        nn.Conv1d(in_ch, out_ch, 1, bias=False), nn.BatchNorm1d(out_ch)
    )
    self.pos_enc = PositionalEncoding(out_ch, max_len=200)
    enc_layer = nn.TransformerEncoderLayer(
        d_model=out_ch, nhead=4, dim_feedforward=256, batch_first=True
    )
    self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)
    self.fc = nn.Linear(out_ch, n_classes)

  def forward(self, x):
    res = self.shortcut(x)
    x = torch.relu(self.inc3(self.inc2(self.inc1(x))) + res)
    x = nn.functional.adaptive_avg_pool1d(x, 125).transpose(1, 2)
    x = self.transformer(self.pos_enc(x)).mean(dim=1)
    return self.fc(x)


# Re-run all 5 models now!
models_to_eval = [
    ('1. Baseline 1D-CNN', SimpleECGCNN(), 'best_simple_cnn.pth'),
    ('2. Inception1D', InceptionNet(), 'inception_seed42.pt'),
    (
        '3. SE-Inception1D (Lead Attn)',
        InceptionNet(use_se=True),
        'se_inception_seed42.pt',
    ),
    (
        '4. SE-Temporal-Inception1D',
        InceptionNet(use_se=True, use_temp_attn=True),
        'se_temporal_inception_seed42.pt',
    ),
    (
        '5. Inception-Transformer',
        InceptionTransformer(),
        'inc_transformer_seed42.pt',
    ),
]

summary, per_class_chap = [], {}
for name, m, ckpt in models_to_eval:
  m.load_state_dict(torch.load(ckpt, map_location=device))
  params = sum(p.numel() for p in m.parameters())

  pv, pt, pc = (
      get_probs(m, val_loader),
      get_probs(m, test_loader),
      get_probs(m, chap_loader),
  )
  ths = np.array([
      np.arange(0.05, 0.96, 0.01)[
          int(
              np.argmax([
                  f1_score(y_val[:, i], pv[:, i] >= t, zero_division=0)
                  for t in np.arange(0.05, 0.96, 0.01)
              ])
          )
      ]
      for i in range(5)
  ])

  ptb_auc = roc_auc_score(y_test, pt, average='macro')
  ptb_f1 = f1_score(y_test, pt >= ths, average='macro', zero_division=0)
  chap_auc = roc_auc_score(y_chap, pc, average='macro')
  chap_f1 = f1_score(y_chap, pc >= ths, average='macro', zero_division=0)
  chap_auprc = average_precision_score(y_chap, pc, average='macro')

  summary.append({
      'Model': name,
      'Params': params,
      'PTB_AUROC': ptb_auc,
      'PTB_F1': ptb_f1,
      'CHAP_AUROC': chap_auc,
      'CHAP_AUPRC': chap_auprc,
      'CHAP_F1': chap_f1,
      'AUROC_Drop': ptb_auc - chap_auc,
  })
  per_class_chap[name] = [
      roc_auc_score(y_chap[:, i], pc[:, i]) for i in range(5)
  ]
  print(
      f'Done: {name} (Params: {params}) -> PTB AUROC: {ptb_auc:.4f} | Chapman'
      f' AUROC: {chap_auc:.4f}'
  )

print('\n================ MASTER CROSS-DATASET COMPARISON ================')
print(pd.DataFrame(summary).set_index('Model').round(4).to_string())

print('\n================ CHAPMAN PER-CLASS AUROC ================')
print(
    pd.DataFrame.from_dict(
        per_class_chap, orient='index', columns=classes
    ).round(4).to_string()
)


Done: 1. Baseline 1D-CNN (Params: 38821) -> PTB AUROC: 0.9189 | Chapman AUROC: 0.8794
Done: 2. Inception1D (Params: 473989) -> PTB AUROC: 0.9277 | Chapman AUROC: 0.8842
Done: 3. SE-Inception1D (Lead Attn) (Params: 498565) -> PTB AUROC: 0.9298 | Chapman AUROC: 0.8829
Done: 4. SE-Temporal-Inception1D (Params: 506886) -> PTB AUROC: 0.9261 | Chapman AUROC: 0.8867
Done: 5. Inception-Transformer (Params: 491141) -> PTB AUROC: 0.9051 | Chapman AUROC: 0.8706

================ MASTER CROSS-DATASET COMPARISON ================
                               Params  PTB_AUROC  PTB_F1  CHAP_AUROC  CHAP_AUPRC  CHAP_F1  AUROC_Drop
Model                                                                                                
1. Baseline 1D-CNN              38821     0.9189  0.7422      0.8794      0.5358   0.4749      0.0395
2. Inception1D                 473989     0.9277  0.7467      0.8842      0.5273   0.4712      0.0436
3. SE-Inception1D (Lead Attn)  498565     0.9298  0.7466      0.8829  

In [20]:
import copy, random, time
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classes = ['CD', 'HYP', 'MI', 'NORM', 'STTC']


# 1. Per-Recording Instance Normalization (Machine/Hospital Independent!)
def instance_norm(X):
  # X shape: (N, 12, 1000) -> har patient ki har lead ko uske khud ke mean/std se scale karo
  mean = X.mean(axis=-1, keepdims=True)
  std = X.std(axis=-1, keepdims=True) + 1e-8
  return ((X - mean) / std).astype(np.float32)


X_train = instance_norm(
    np.transpose(np.load('processed_data/X_train.npy'), (0, 2, 1))
)
y_train = np.load('processed_data/y_train.npy')

X_val = instance_norm(
    np.transpose(np.load('processed_data/X_val.npy'), (0, 2, 1))
)
y_val = np.load('processed_data/y_val.npy')

X_test = instance_norm(
    np.transpose(np.load('processed_data/X_test.npy'), (0, 2, 1))
)
y_test = np.load('processed_data/y_test.npy')

# Chapman (Untouched External Test Set - same instance_norm!)
X_chap = instance_norm(np.load('processed_chapman/X_chapman.npy'))
y_chap = np.load('processed_chapman/y_chapman.npy')

train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train), torch.tensor(y_train, dtype=torch.float32)
    ),
    batch_size=64,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val)), batch_size=128, shuffle=False
)
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test)), batch_size=128, shuffle=False
)
chap_loader = DataLoader(
    TensorDataset(torch.tensor(X_chap)), batch_size=128, shuffle=False
)


# 2. Training-Only ECG Augmentation
def augment_ecg(xb):
  # Random amplitude scaling (0.8x to 1.2x)
  scale = torch.empty((xb.size(0), 1, 1), device=xb.device).uniform_(0.8, 1.2)
  xb = xb * scale
  # Random baseline shift
  shift = torch.empty((xb.size(0), 12, 1), device=xb.device).uniform_(-0.1, 0.1)
  xb = xb + shift
  # Light Gaussian noise
  xb = xb + 0.03 * torch.randn_like(xb)
  return xb


# 3. Train Upgraded SE-Temporal-Inception1D
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

model = InceptionNet(use_se=True, use_temp_attn=True).to(device)
# Mild pos_weight = sqrt(neg/pos) taaki probabilities calibrated rahein!
pos = y_train.sum(0)
mild_weight = torch.tensor(
    np.sqrt((len(y_train) - pos) / pos), dtype=torch.float32
).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=mild_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=25, eta_min=1e-5
)

best_auc, best_state, bad = -1, None, 0
for ep in range(1, 26):
  t0 = time.time()
  model.train()
  tot = 0.0
  for xb, yb in train_loader:
    xb, yb = augment_ecg(xb.to(device)), yb.to(device)
    optimizer.zero_grad()
    loss = criterion(model(xb), yb)
    loss.backward()
    optimizer.step()
    tot += loss.item()
  scheduler.step()

  model.eval()
  with torch.no_grad():
    pv = np.vstack([torch.sigmoid(model(xb.to(device))).cpu().numpy() for (xb,) in val_loader])
  val_auc = roc_auc_score(y_val, pv, average='macro')
  print(
      f'ep {ep:02d} | loss {tot/len(train_loader):.4f} | val AUROC'
      f' {val_auc:.4f} | {time.time()-t0:.0f}s'
  )

  if val_auc > best_auc:
    best_auc, best_state, bad = val_auc, copy.deepcopy(model.state_dict()), 0
  else:
    bad += 1
    if bad >= 7:
      print('early stop')
      break

model.load_state_dict(best_state)
torch.save(model.state_dict(), 'se_temporal_inception_robust.pt')

# 4. Evaluate on PTB-XL Test & Untouched Chapman!
model.eval()
with torch.no_grad():
  pv = np.vstack([torch.sigmoid(model(xb.to(device))).cpu().numpy() for (xb,) in val_loader])
  pt = np.vstack([torch.sigmoid(model(xb.to(device))).cpu().numpy() for (xb,) in test_loader])
  pc = np.vstack([torch.sigmoid(model(xb.to(device))).cpu().numpy() for (xb,) in chap_loader])

ths = np.array([
    np.arange(0.05, 0.96, 0.01)[
        int(
            np.argmax([
                f1_score(y_val[:, i], pv[:, i] >= t, zero_division=0)
                for t in np.arange(0.05, 0.96, 0.01)
            ])
        )
    ]
    for i in range(5)
])

print('\n================ ROBUST TRAINING RESULTS ================')
print('PTB-XL Fold 9 Thresholds:', dict(zip(classes, ths.round(2))))
print(
    f'PTB-XL Test  -> Macro-AUROC: {roc_auc_score(y_test, pt, average="macro"):.4f} |'
    f' Macro-F1: {f1_score(y_test, pt >= ths, average="macro"):.4f}'
)
print(
    f'Chapman Test -> Macro-AUROC: {roc_auc_score(y_chap, pc, average="macro"):.4f} |'
    f' Macro-AUPRC: {average_precision_score(y_chap, pc, average="macro"):.4f} |'
    f' Macro-F1: {f1_score(y_chap, pc >= ths, average="macro"):.4f}'
)


ep 01 | loss 0.4853 | val AUROC 0.8902 | 41s
ep 02 | loss 0.4176 | val AUROC 0.8962 | 40s
ep 03 | loss 0.3964 | val AUROC 0.9036 | 42s
ep 04 | loss 0.3811 | val AUROC 0.9033 | 42s
ep 05 | loss 0.3704 | val AUROC 0.9057 | 39s
ep 06 | loss 0.3581 | val AUROC 0.9090 | 38s
ep 07 | loss 0.3503 | val AUROC 0.9084 | 40s
ep 08 | loss 0.3403 | val AUROC 0.9070 | 40s
ep 09 | loss 0.3338 | val AUROC 0.9115 | 39s
ep 10 | loss 0.3238 | val AUROC 0.9080 | 48s
ep 11 | loss 0.3126 | val AUROC 0.9058 | 48s
ep 12 | loss 0.2999 | val AUROC 0.9116 | 49s
ep 13 | loss 0.2881 | val AUROC 0.9077 | 50s
ep 14 | loss 0.2763 | val AUROC 0.9059 | 50s
ep 15 | loss 0.2605 | val AUROC 0.9070 | 49s
ep 16 | loss 0.2480 | val AUROC 0.9047 | 49s
ep 17 | loss 0.2308 | val AUROC 0.9080 | 48s
ep 18 | loss 0.2108 | val AUROC 0.9026 | 48s
ep 19 | loss 0.1942 | val AUROC 0.9050 | 48s
early stop

================ ROBUST TRAINING RESULTS ================
PTB-XL Fold 9 Thresholds: {'CD': np.float64(0.48), 'HYP': np.float64(0.4), 